# 3. Data Cleaning and Preparation

Real-world datasets are rarely clean. This notebook covers the essential steps of **data preparation**:
- Detecting and handling missing values
- Removing duplicates
- Type conversion
- Encoding categorical variables (one-hot, label encoding)
- Feature engineering
- Feature scaling (StandardScaler, MinMaxScaler)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

## 3.1 Detecting Missing Values

The first step is always to assess **what is missing and how much**. Use `.isnull().sum()` for counts and `.isnull().mean()` for percentages.

In [ ]:
titanic = sns.load_dataset('titanic')
print(f"Shape: {titanic.shape}")
print(f"\nMissing values:")
print(titanic.isnull().sum())
print(f"\nMissing percentages:")
print((titanic.isnull().mean() * 100).round(1))

## 3.2 Handling Missing Data

Common strategies:
- **Numerical columns**: fill with **median** (robust to outliers)
- **Categorical columns**: fill with **mode** (most frequent value)
- **Columns with > 70% missing**: **drop** them entirely

In [ ]:
df = titanic.copy()

# Fill age with median
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)
print(f"Age filled with median: {median_age}")

# Fill embarked with mode
mode_embarked = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(mode_embarked)
print(f"Embarked filled with mode: {mode_embarked}")

# Drop deck (too many missing)
df = df.drop(columns=['deck'])
print(f"\nRemaining missing values: {df.isnull().sum().sum()}")

## 3.3 Duplicates and Type Conversion

Always check for duplicate rows and ensure columns have the correct data types.

In [ ]:
print(f"Duplicates found: {df.duplicated().sum()}")
df = df.drop_duplicates()

df['pclass'] = df['pclass'].astype('category')
print(f"pclass type: {df['pclass'].dtype}")

## 3.4 Encoding Categorical Variables

Machine learning models require numerical input. Two common approaches:
- **One-hot encoding**: creates binary columns for each category (use for nominal variables)
- **Label encoding**: maps categories to integers (use for ordinal variables or tree-based models)

In [ ]:
# One-hot encoding
embarked_dummies = pd.get_dummies(df['embarked'], prefix='embarked', dtype=int)
print("One-hot encoding (embarked):")
print(embarked_dummies.head(3))

# Label encoding
le = LabelEncoder()
df['sex_encoded'] = le.fit_transform(df['sex'])
print(f"\nLabel encoding (sex): {dict(zip(le.classes_, le.transform(le.classes_)))}")

## 3.5 Feature Engineering

Creating new features from existing ones can improve model performance. Domain knowledge is key.

In [ ]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)

print(f"Family size stats:\n{df['family_size'].describe().round(2)}")
print(f"\nAlone passengers: {df['is_alone'].sum()} ({df['is_alone'].mean():.1%})")

## 3.6 Feature Scaling

Many algorithms (KNN, SVM, neural networks) are sensitive to feature magnitudes.
- **StandardScaler**: zero mean, unit variance (z-scores)
- **MinMaxScaler**: scales to [0, 1]

In [ ]:
scaler_std = StandardScaler()
scaler_mm = MinMaxScaler()

age_std = scaler_std.fit_transform(df[['age']])
age_mm = scaler_mm.fit_transform(df[['age']])

print(f"Original   : mean={df['age'].mean():.2f}, std={df['age'].std():.2f}")
print(f"Standard   : mean={age_std.mean():.4f}, std={age_std.std():.4f}")
print(f"MinMax     : min={age_mm.min():.4f}, max={age_mm.max():.4f}")

## Key Takeaways

1. **Always start** by assessing missing values and their percentages
2. Use **median** for numerical imputation (robust to outliers)
3. **One-hot encode** nominal categories; **label encode** ordinal ones
4. **Feature engineering** leverages domain knowledge to create informative variables
5. **Scale features** before distance-based or gradient-based algorithms